# Time Series

In [ ]:
import numpy as np
import json
import pandas as pd
from pprint import pprint
from pathlib import Path

## Roster Selection

First, from the results of the previous notebook, we will select our roster with a criteria heuristic I unrigorously made up.

In [ ]:
target_dir = Path("../data/processed")
metrics = ["score", "wc", "favorites", "dropped", "forum"]
data = {}

for metric in metrics:
    file_path = target_dir / f"{metric}_predictions.json"
    try:
        with file_path.open("r", encoding="utf-8") as file:
            data[metric] = json.load(file)
    except FileNotFoundError:
        print(f"Error: The file at {file_path} does not exist.")
    except json.JSONDecodeError:
        print(f"Error: The file at {file_path} contains invalid JSON formatting.")

In [ ]:
score_data = data.get('score', {})
wc_data = data.get('wc', {})
favorites_data = data.get('favorites', {})
drop_data = data.get('dropped', {})
forum_data = data.get('forum', {})

In [ ]:
criteria = {}

for key in score_data:
    criteria[key] = wc_data.get(key, [])[0] * (5 / 9) + forum_data.get(key, [])[0] / 6 + score_data.get(key, [])[0] * (2 / 9) - drop_data.get(key, [])[0] / 9 + favorites_data.get(key, [])[0] / 6

sorted_criteria = dict(sorted(criteria.items(), key=lambda item: item[1], reverse=True))
pprint(sorted_criteria, indent=4, sort_dicts=False)

We will use Kalman filters to create our forecasting model. Refer to the documentation for more clarity on the algorithm.

In [ ]:
class KalmanFilter:
    def __init__(self, 
                 transition_mat,
                 observation_mat,
                 noise_mat,
                 measured_covs,
                 measurements):
        self.transition_mat = transition_mat # state transition matrix
        self.observation_mat = observation_mat # observation matrix, will most likely be I
        self.noise_mat = noise_mat # process noise
        self.measured_covs = measured_covs # covariance matrices across days; note that day 0 is a prediction from the neural network
        self.measurements = measurements # z variables in the documentation, 7 by n array

    def check(self):
        # first check, transition matrix
        if self.transition_mat.shape[0] != self.transition_mat.shape[1]:
            return False
        else:
            dim = self.transition_mat.shape[0]

        # second check, observation matrix
        if self.observation_mat.shape[0] != self.observation_mat.shape[1] or self.observation_mat.shape[0] != dim:
            return False

        # third check, noise matrix
        if self.noise_mat.shape[0] != self.noise_mat.shape[1] or self.noise_mat.shape[0] != dim:
            return False

        # fourth check, measured covariance matrices
        measured_days = self.measurements.shape[0]
        if self.measured_covs.shape[0] != measured_days or self.measured_covs.shape[1] != self.measured_covs.shape[2] or self.measured_covs.shape[1] != dim:
            return False

    def make_predictions(self):
        measured_days = self.measurements.shape[0] # starts with day 0
        num_features = self.measurements.shape[1]

        # more matrix initializations
        uncertainty_mats = np.zeros((measured_days + 1, measured_days, num_features, num_features)) # first two dimensions for the two subscripts, and then each entry is a two by two square matrix
        state_mats = np.zeros((measured_days + 1, measured_days, num_features)) # first two dimensions for the two subscripts, and the entry is a 1D array
        kalman_gains = np.zeros((measured_days, num_features, num_features))  # 1D array of 2 by 2 matrices

        state_mats[0][0] = self.measurements[0]
        
        for i in range(measured_days):
            uncertainty_mats[i][i] = self.measured_covs[i]

        # per-day updates
        for day in range(measured_days):
            state_mats[day+1][day] = self.transition_mat @ state_mats[day][day]

            uncertainty_mats[day+1][day] = self.transition_mat @ uncertainty_mats[day][day] @ self.transition_mat.T + self.noise_mat

            if day < measured_days - 1:
                kalman_gains[day+1] = uncertainty_mats[day+1][day] @ self.observation_mat @ np.linalg.inv(self.observation_mat @ uncertainty_mats[day+1][day] @ self.observation_mat.T + uncertainty_mats[day+1][day+1])

                state_mats[day+1][day+1] = state_mats[day+1][day] + kalman_gains[day+1] @ (self.measurements[day+1] - self.observation_mat @ state_mats[day+1][day])

        # return prediction list, day by day
        return [state_mats[day+1][day] for day in range(measured_days)]
    

## Test Run

From the example: https://kalmanfilter.net

In [ ]:
measurements = np.array([[10_000, 200], 
                         [11_020, 202]])
covs = np.array([[[16, 0], 
                  [0, 0.25]], 
                  [[36, 0], 
                   [0, 2.25]]])
transition = np.array([[1, 5], 
                       [0, 1]])
noise = np.array([[6.25, 2.5], 
                  [2.5, 1]])
observation = np.eye(2)

test_kalman = KalmanFilter(transition, observation, noise, covs, measurements)
test_prediction = test_kalman.make_predictions()

print(test_prediction)

In [ ]:
target_dir = Path("../data/processed")
file_path = target_dir / "current_stats.json"

try:
    with file_path.open("r", encoding="utf-8") as file:
        current_stats = json.load(file)
except FileNotFoundError:
    print(f"Error: The file at {file_path} does not exist.")
except json.JSONDecodeError:
    print(f"Error: The file at {file_path} contains invalid JSON formatting.")

In [ ]:
roster_titles = []

## WC Raw Count Forecasting (for Acing)

In [ ]:
roster_wcs = {}
initial_speed = 1000

for title in roster_titles:
    temp_measurements = np.zeros((len(current_stats), 2))

    for day in current_stats:
        temp_measurements[day][0] = current_stats.get(day, {}).get(title, {}).get("wc_raw", 0)
        temp_measurements[day][1] = current_stats.get(day, {}).get(title, {}).get("wc_raw_dot", 0)

    temp_observation = np.eye(2)
    temp_noise = np.array([[6.25, 2.5], 
                                  [2.5, 1]])
    temp_covs = np.zeros((len(current_stats), 2, 2)) # CHANGE THIS TO PREDICTED SAMPLE COVARIANCES
    temp_transition = np.array([[1, 5], 
                                [0, 1]])
    
    temp_kalman = KalmanFilter(temp_transition, temp_observation, temp_noise, temp_covs, temp_measurements)
    roster_wcs[title] = temp_kalman.make_predictions()